In [49]:
import pandas as pd
amazon=pd.read_parquet("parquet/amazon_new.parquet")
amazon.drop(["author_norm","title_norm"],axis=1,inplace=True)
goodreads=pd.read_parquet("parquet/goodreads_new.parquet")
# We are dropping columns that are not needed for data fusion. These two columns only exist in goodreads dataset.
goodreads.drop(["book_format","edition","title_norm","author_norm"],axis=1,inplace=True)
metabooks=pd.read_parquet("parquet/metabooks_new.parquet")
metabooks.drop(["author_norm","title_norm"],axis=1,inplace=True)


In [50]:
display(amazon.head(1).style.set_caption("Amazon"))
display(goodreads.head(1).style.set_caption("Goodreads"))
display(metabooks.head(1).style.set_caption("Metabooks"))


,id,title,author,publish_year,publisher,isbn_clean
0,amazon_1,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,0195153448


,id,title,author,rating,rating_number,language,genres,page_count,publisher,publish_year,price,isbn_clean
0,goodreads_1,The Hunger Games,Suzanne Collins,4.330000,6376780,English,"['Young Adult', 'Fiction', 'Dystopia', 'Fantasy', 'Science Fiction', 'Romance', 'Adventure', 'Teen', 'Post Apocalyptic', 'Action']",374,Scholastic Press,2008,5.090000,0439023483


,id,title,author,rating,rating_number,language,genres,publisher,page_count,price,publish_year,isbn_clean
0,metabooks_1,Alvis: The Story of the Red Triangle,Kenneth Day,4.100000,3,English,"['Engineering', 'Transportation', 'Automotive']",Haynes Pubns,400,112.300003,,1844255247


In [51]:
common_isbns = (
    set(metabooks["isbn_clean"].dropna())
    & set(goodreads["isbn_clean"].dropna())
    & set(amazon["isbn_clean"].dropna())
)

len(common_isbns)
first_isbns = sorted(common_isbns)[:20]
samples = []
for isbn in first_isbns:
    rows = []
    for source, df in [("Metabooks", metabooks), ("Goodreads", goodreads), ("Amazon", amazon)]:
        match = df[df["isbn_clean"] == isbn].copy()
        match["source"] = source
        rows.append(match)
    samples.append(pd.concat(rows, ignore_index=True))

sample_matches = pd.concat(samples, ignore_index=True)

display(sample_matches)
out_path = "common_isbn_sample.csv"
sample_matches.to_csv(out_path, index=False)
print(f"Saved {len(sample_matches)} rows to {out_path}")


/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_24053/24089234.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  samples.append(pd.concat(rows, ignore_index=True))


,id,title,author,rating,rating_number,language,genres,publisher,page_count,price,publish_year,isbn_clean,source
0,metabooks_93909,Complete Works of Oscar Wilde (Collins Classics),Oscar Wilde,4.60,1028,English,"['Literature', 'Fiction', 'Dramas', 'Plays']",Collins,1216,92.360001,<NA>,0007144350,Metabooks
1,goodreads_5479,Complete Works of Oscar Wilde,"Oscar Wilde, Merlin Holland (Introduction)",4.48,13921,English,"['Classics', 'Fiction', 'Poetry', 'Plays', 'Short Stories', 'Literature', 'Drama', 'Humor', 'Essays', '19th Century']",HarperCollins Publishers,1270,31.6,2003,0007144350,Goodreads
2,amazon_42849,Collins Complete Works of Oscar Wilde (Collins Classics S.),Oscar Wilde,NaN,<NA>,<NA>,<NA>,HarperCollins Publishers,<NA>,<NA>,2003,0007144350,Amazon
3,metabooks_163904,Unless: A Novel,Carol Shields,4.30,363,English,"['Literature', 'Fiction', 'Genre Fiction']",Harper Perennial,336,2.99,<NA>,0007154615,Metabooks
4,goodreads_12051,Unless,Carol Shields,3.63,12837,English,"['Fiction', 'Canada', 'Contemporary', 'Literary Fiction', 'Novels', 'Canadian Literature', 'Literature', 'Adult Fict...",Fourth Estate,320,3.12,2003,0007154615,Goodreads
5,amazon_3233,Unless : A Novel,Carol Shields,NaN,<NA>,<NA>,<NA>,Perennial,<NA>,<NA>,2003,0007154615,Amazon
6,metabooks_34322,Finding Fish: A Memoir,Antwone Quenton Fisher,4.70,859,English,"['Biographies', 'Memoirs', 'Arts', 'Literature']",William Morrow Paperbacks,352,9.39,<NA>,0060007788,Metabooks
7,goodreads_11877,Finding Fish,"Antwone Quenton Fisher, Mim Eichler Rivas",4.21,3470,English,"['Nonfiction', 'Memoir', 'Biography', 'African American', 'Autobiography', 'Biography Memoir', 'Abuse', 'Fostering',...",Perennial,352,1.65,2001,0060007788,Goodreads
8,amazon_50944,Finding Fish: A Memoir,Antwone Q. Fisher,NaN,<NA>,<NA>,<NA>,Perennial,<NA>,<NA>,2001,0060007788,Amazon
9,metabooks_239401,Dharma Punx,Noah Levine,4.60,759,English,"['Arts', 'Photography', 'Music']",HarperOne,272,10.49,<NA>,0060008954,Metabooks


Saved 60 rows to common_isbn_sample.csv


In [8]:
display(pd.Series(amazon.info(), name="Amazon columns"))
display(pd.Series(goodreads.info(), name="Goodreads columns"))
display(pd.Series(metabooks.info(), name="Metabooks columns"))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            271360 non-null  object
 1   title         271360 non-null  string
 2   author        271358 non-null  string
 3   publish_year  271357 non-null  Int16 
 4   publisher     271358 non-null  string
 5   isbn_clean    271360 non-null  string
 6   title_norm    271360 non-null  object
 7   author_norm   271360 non-null  object
dtypes: Int16(1), object(3), string(4)
memory usage: 15.3+ MB


Series([], Name: Amazon columns, dtype: object)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52478 entries, 0 to 52477
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             52478 non-null  object 
 1   title          52478 non-null  string 
 2   author         52478 non-null  string 
 3   rating         52478 non-null  float64
 4   rating_number  52478 non-null  Int64  
 5   language       52478 non-null  string 
 6   genres         52478 non-null  string 
 7   book_format    52478 non-null  string 
 8   edition        52478 non-null  string 
 9   page_count     49967 non-null  Int32  
 10  publisher      52478 non-null  string 
 11  publish_year   50801 non-null  Int16  
 12  price          38101 non-null  float64
 13  isbn_clean     52478 non-null  string 
 14  title_norm     52478 non-null  object 
 15  author_norm    52478 non-null  object 
dtypes: Int16(1), Int32(1), Int64(1), float64(2), object(3), string(8)
memory usage: 6.1+ MB


Series([], Name: Goodreads columns, dtype: object)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 535159 entries, 0 to 535158
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id             535159 non-null  object 
 1   title          535159 non-null  string 
 2   author         535159 non-null  string 
 3   rating         535159 non-null  float64
 4   rating_number  535159 non-null  Int64  
 5   language       531658 non-null  string 
 6   genres         501318 non-null  string 
 7   publisher      522621 non-null  string 
 8   page_count     483017 non-null  Int32  
 9   price          467654 non-null  Float32
 10  publish_year   250947 non-null  Int16  
 11  isbn_clean     535159 non-null  string 
 12  title_norm     535159 non-null  object 
 13  author_norm    535159 non-null  object 
dtypes: Float32(1), Int16(1), Int32(1), Int64(1), float64(1), object(3), string(6)
memory usage: 52.1+ MB


Series([], Name: Metabooks columns, dtype: object)